# TT-LLM: TPU Edition

Same pipeline (compress → SFT → KD → eval) running on **Google TPU** via PyTorch/XLA.

TPU advantages:
- More memory than a single GPU (16-32GB on Colab TPU v2)
- Faster matrix multiplications
- No autocast needed — model loaded in bf16 directly


## 0. Setup

In [ ]:
import os

if not os.path.exists('/content/TT_LLM'):
    !git clone https://github.com/WaelRabah/TT_LLM.git /content/TT_LLM

%cd /content/TT_LLM
!git pull

# Install torch-xla for TPU support (must match torch version)
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = 'cu118' if 'cu118' in torch.__version__ else 'cu121'
!pip install torch-xla -f https://storage.googleapis.com/tpu-pytorch/wheels/colab/torch_xla-{TORCH_VERSION}-cp311-en.html -q || \
 !pip install torch-xla -q

!pip install -e ".[colab]" -q

### Verify TPU is available

In [ ]:
import torch
import torch_xla.core.xla_model as xm

device = xm.xla_device()
print(f'PyTorch: {torch.__version__}')
print(f'XLA device: {device}')
print(f'TPU cores: {xm.get_xla_supported_devices()}')

**If the cell above fails**, switch the Colab runtime:
- Runtime → Change runtime type → TPU v2
- Then Runtime → Restart session and re-run from the top

## 1. Config

In [ ]:
METHOD = 'sola'          # 'tt_llm' or 'sola'
COMPRESSION_PCT = 30      # remove 30% of params
SFT_EPOCHS = 3
KD_EPOCHS = 2
EVAL_LIMIT = 500         # examples per benchmark (None = full)

## 2. Run Full Pipeline

One call does everything. The pipeline auto-detects TPU via `torch_xla`.

In [ ]:
from tt_llm.pipeline import run_pipeline

results = run_pipeline(
    method=METHOD,
    compression_pct=COMPRESSION_PCT,
    sft_epochs=SFT_EPOCHS,
    kd_epochs=KD_EPOCHS,
    eval_limit=EVAL_LIMIT,
)

## 3. Results

In [ ]:
import matplotlib.pyplot as plt

# Plot SFT loss
sft = results.get('sft_metrics', {})
if sft.get('train_loss'):
    plt.figure(figsize=(10, 4))
    plt.plot(sft['train_loss'], alpha=0.3, label='step loss')
    window = 50
    if len(sft['train_loss']) > window:
        rolling = [sum(sft['train_loss'][max(0,i-window):i])/min(i,window)
                   for i in range(1, len(sft['train_loss'])+1)]
        plt.plot(rolling, label=f'rolling ({window})', linewidth=2)
    plt.xlabel('Step'); plt.ylabel('Loss')
    plt.title('SFT Training Loss'); plt.legend()
    plt.show()

# Plot KD losses
kd = results.get('kd_metrics', {})
if kd.get('train_loss'):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(kd['train_loss'], alpha=0.3)
    axes[0].set_title('Total Loss')
    axes[1].plot(kd['train_kd_loss'], alpha=0.3, color='orange')
    axes[1].set_title('KD Loss (KL)')
    axes[2].plot(kd['train_ce_loss'], alpha=0.3, color='green')
    axes[2].set_title('CE Loss')
    for ax in axes: ax.set_xlabel('Step')
    plt.tight_layout()
    plt.show()

## 4. Save to Google Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./checkpoints/final_compressed /content/drive/MyDrive/tt_llm_models/